In [2]:
print('DU DU DU MAX VERSTAPPEN')

DU DU DU MAX VERSTAPPEN


In [3]:
import pandas as pd

def merge_driver_weather(
    driver_df,
    weather_df,
    output_csv="final_merged_2024_data.csv"
):

    """
    Merge driver and weather datasets using nearest
    backward timestamp alignment within the same
    GP and SessionType.
    """

    # Create copies
    driver = driver_df.copy()
    weather = weather_df.copy()

    # Standardize column names to match 2023 schema
    driver = driver.rename(
        columns={
            "Session": "SessionType",
            "Time": "DriverTime"
        }
    )

    weather = weather.rename(
        columns={
            "EventName": "GP",
            "Session": "SessionType",
            "Time": "WeatherTime"
        }
    )

    # Convert time columns
    driver["DriverTime"] = pd.to_timedelta(
        driver["DriverTime"]
    )

    weather["WeatherTime"] = pd.to_timedelta(
        weather["WeatherTime"]
    )

    # Sort before merge
    driver = driver.sort_values(
        ["GP", "SessionType", "Driver", "DriverTime"]
    )

    weather = weather.sort_values(
        ["GP", "SessionType", "WeatherTime"]
    )

    # Precompute weather groups
    weather_groups = weather.groupby(
        ["GP", "SessionType"]
    )

    merged_chunks = []

    # Merge GP-session wise
    for (gp, session), d_group in driver.groupby(
        ["GP", "SessionType"]
    ):

        if (gp, session) not in weather_groups.groups:
            continue

        w_group = weather_groups.get_group(
            (gp, session)
        )

        # Ensure sorted inside groups
        d_group = d_group.sort_values(
            "DriverTime"
        ).reset_index(drop=True)

        w_group = w_group.sort_values(
            "WeatherTime"
        ).reset_index(drop=True)

        # Temporal nearest merge
        merged = pd.merge_asof(
            d_group,
            w_group,
            left_on="DriverTime",
            right_on="WeatherTime",
            direction="backward",
            tolerance=pd.Timedelta("5min")
        )
        # Reattach group context
        merged["GP"] = gp
        merged["SessionType"] = session

        merged_chunks.append(merged)

    # Combine all chunks
    merged_df = pd.concat(
        merged_chunks,
        ignore_index=True
    )

    # Save merged dataset
    merged_df.to_csv(
        output_csv,
        index=False
    )

    print(f"Merge complete. Saved to {output_csv}")

    return merged_df

In [5]:
sorted_driver = pd.read_csv("C:/Users/VANSH/Formula1/datasets/2025/sorted/driver_sorted.csv")
sorted_weather = pd.read_csv("C:/Users/VANSH/Formula1/datasets/2025/sorted/weather_sorted.csv")

sorted_driver.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate', 'RoundNumber', 'GP', 'Session',
       'Date'],
      dtype='object')